In [1]:
import numpy as np
import json

# Carrega o JSON
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/datas_train_mlp_v4.json', 'r') as f:
    index_data = json.load(f)

X_bruto = [] # Agora conterá apenas [Vg, Vd]
Y_labels_ponto = [] # Mantém ln(|Id|)

for item in index_data:
    npz_path = item["npz_path"]
    data = np.load(npz_path, allow_pickle=True)
    
    V_model = data["V"].ravel()
    I_model = data["I"].ravel()
    V_fixed = item["fixed_voltage_simulated"] 
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = data['meta'].tolist()['is_transfer']
    
    for V_exp, I_exp in zip(V_model, I_model):
        if is_transfer:
            V_G = V_exp
            V_D = V_fixed
        else:
            V_G = V_fixed
            V_D = V_exp
        
        # --- ADAPTAÇÃO: Lendo dados diretamente sem transformações ---
        # Passamos apenas as variáveis independentes brutas
        X_ponto = [V_G, V_D]
        
        # Label Engineering (Mantido para manter a escala logarítmica, comum em OTFTs)
        I_abs = np.abs(I_exp)
        Id_min = 1e-30 
        Y_label = np.log(max(I_abs, Id_min)) 
        
        X_bruto.append(X_ponto)
        Y_labels_ponto.append(Y_label)
        
# Conversão Final
X = np.array(X_bruto, dtype=np.float32)
Y = np.array(Y_labels_ponto, dtype=np.float32).reshape(-1, 1)

print(f"Formato de X: {X.shape}") # Resultado esperado: (n_pontos, 2)

Formato de X: (50000, 2)


In [2]:
X.shape # será (N_total_pontos, 5)
Y.shape # será (N_total_pontos, 1)


(50000, 1)

In [3]:
from sklearn.preprocessing import StandardScaler

# Padronizar X (crucial para convergência)
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

## **EXP 1**

In [6]:
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(256, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(128, activation='tanh', name='HL2'),
    Dense(64, activation='tanh', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=500, batch_size=32, validation_split=0.2)

#Salvar o modelo treinado e o scaler_X
base_path = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v4/exp1"
model.save(f"{base_path}/modelo_exp1.keras")
joblib.dump(scaler_X, f"{base_path}/scaler_X_exp1.pkl")
joblib.dump(Y, f"{base_path}/scaler_Y_exp1.pkl")

2026-01-26 10:50:50.053353: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-26 10:50:50.144877: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-26 10:50:53.454136: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1769435454.676119 1912930 

Epoch 1/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.3111 - mae: 0.2495 - val_loss: 0.0882 - val_mae: 0.2761
Epoch 2/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0012 - mae: 0.0246 - val_loss: 0.0335 - val_mae: 0.1682
Epoch 3/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 5.7587e-04 - mae: 0.0161 - val_loss: 0.0264 - val_mae: 0.1537
Epoch 4/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 5.5301e-04 - mae: 0.0162 - val_loss: 0.0209 - val_mae: 0.1334
Epoch 5/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0012 - mae: 0.0201 - val_loss: 0.0385 - val_mae: 0.1760
Epoch 6/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 6.9580e-04 - mae: 0.0156 - val_loss: 0.0046 - val_mae: 0.0523
Epoch 7/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 8.6204e-04 - mae: 0.0176 - val_loss: 0.0108 - val_mae: 0.0898
Epoch 8/500
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 5.8725e-04 - mae: 0.0143 - val_loss: 0.0105 - val_mae: 0.0961
Epoch 9/500


['/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v4/exp1/scaler_Y_exp1.pkl']

In [11]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from typing import Optional, Tuple, Callable
# Importar a função 'Callable' para indicar o tipo de argumento do modelo

def plot_curve_comparison(
    V_data: np.ndarray, 
    I_real: np.ndarray,
    V_fixed: float,
    is_transfer: bool,
    mlp_model: Optional[Tuple[Callable, object]] = None,
    yscale: str = "log", 
    title: Optional[str] = None
):
    """
    Plota a curva real (I_real vs V_data) e opcionalmente a previsão da MLP.

    Parâmetros
    ----------
    V_data : np.ndarray
        Vetor de tensão (eixo X) lido do CSV (Vg para transferência, Vd para saída).
    I_real : np.ndarray
        Vetor de corrente real (eixo Y) lido do CSV.
    V_fixed : float
        Tensão fixa da curva (Vd para transferência, Vg para saída).
    is_transfer : bool
        True se for curva de transferência (Vgs vs Ids), False se for curva de saída (Vds vs Ids).
    mlp_model : tuple (Callable, object) opcional
        Tupla contendo (função_de_preparação_X, modelo_MLP_treinado, scaler_X).
        Permite que a função gere as previsões internamente.
    yscale : str {"log", "linear", "logarithmic"}
        Tipo de escala no eixo Y.
    title : str
        Título opcional do gráfico.
    """

    fig = go.Figure()
    
    # Normaliza a escala Y para checagem
    y_scale = yscale.lower()
    is_log_scale = y_scale in ("log", "logarithmic")
    y_axis_title = "Corrente Absoluta (A)" if is_log_scale else "Corrente (A)"
    
    # -------------------------------
    # PREPARAÇÃO E PLOTAGEM DA CURVA REAL
    # -------------------------------
    
    # Aplicar a correção Logarítmica apenas para o plot, se necessário
    I_plot_real = np.abs(I_real)
    if is_log_scale:
        I_plot_real[I_plot_real <= 0] = 1e-30
    
    fig.add_trace(go.Scatter(
        x=V_data, 
        y=I_plot_real, 
        mode='lines+markers', 
        name="Real (Dados CSV)",
        line=dict(color='blue')
    ))

    # -------------------------------
    # PREVISÃO E PLOTAGEM DA MLP (Se fornecida)
    # -------------------------------
    if mlp_model is not None:
        # Desempacota as ferramentas necessárias
        prepare_X_func, model, scaler_X = mlp_model
        
        # 1. Preparar features X para a curva inteira
        X_test = prepare_X_func(V_data, V_fixed, is_transfer)
        
        # 2. Padronizar X (CRUCIAL: Usar o scaler treinado)
        X_test_scaled = scaler_X.transform(X_test)
        
        # 3. Prever ln(|Id|)
        Y_pred_ln = model.predict(X_test_scaled).ravel()
        
        # 4. Converter para Corrente (|Id|)
        I_pred_abs = np.exp(Y_pred_ln)
        
        # 5. Preparar para Plotagem Logarítmica (se necessário)
        I_plot_pred = I_pred_abs.copy()
        if is_log_scale:
            I_plot_pred[I_plot_pred <= 0] = 1e-30 
        
        fig.add_trace(go.Scatter(
            x=V_data, 
            y=I_plot_pred, 
            mode='lines', 
            name="Previsão MLP",
            line=dict(color='red', dash='dash')
        ))
        
        # Calcula MAE na escala logarítmica para referência
        I_real_ln = np.log(I_plot_real)
        I_pred_ln_for_metric = np.log(I_plot_pred)
        mae = np.mean(np.abs(I_real_ln - I_pred_ln_for_metric))
        
        if title is None:
             title = f"Curva de Teste vs. Previsão MLP (V_fixed={V_fixed}V, MAE_ln={mae:.3f})"
        else:
             title += f" (MAE_ln={mae:.3f})"


    fig.update_layout(
        title=title if title is not None else "Curva Real (CSV)",
        xaxis_title="V_G (V)" if is_transfer else "V_D (V)",
        yaxis_title=y_axis_title,
        yaxis_type="log" if is_log_scale else "linear",
        template="plotly_dark",
        legend_title="Curvas"
    )

    fig.show()
    return

In [9]:
import numpy as np

def prepare_mlp_features(V_data: np.ndarray, V_fixed: float, is_transfer: bool) -> np.ndarray:
    """
    Gera o array de features X para a MLP a partir dos vetores de tensão de uma curva.
    
    Ajustado para o novo cenário: Retorna apenas as variáveis brutas [Vg, Vd].
    """
    
    # V_data é o vetor do eixo X (V_exp). V_fixed é a tensão constante da simulação.
    
    if is_transfer:
        # Transfer: Eixo X é Vg, Tensão fixa é Vds
        V_G = V_data
        V_D = np.full_like(V_data, V_fixed)
    else: 
        # Output: Eixo X é Vds, Tensão fixa é Vg
        V_G = np.full_like(V_data, V_fixed)
        V_D = V_data

    # Construindo a matriz com apenas 2 colunas
    # Isso deve bater com o input_shape=(2,) da sua rede neural
    X_features = np.column_stack([
        V_G,
        V_D
    ])
    
    return X_features

In [9]:
# Carregar json de teste
with open('/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json', 'r') as f:
    infer_data = json.load(f)
for item in infer_data:
    npz_path = item["npz_path"]
    csv_path = item["csv_path"]
    v_fixed = item["fixed_voltage_simulated"]
    params = item["params"]
    
    # Determinar se é curva de Transferência ou Saída
    is_transfer = 'transfer' in os.path.basename(npz_path).lower()
    
    # Carregar dados do CSV
    df = pd.read_csv(csv_path)
    V_real = df.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
    I_real = df.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    
    # 2. Preparar o pacote de ferramentas para plotagem
    pacote_ferramentas = (prepare_mlp_features, model, scaler_X)
    
    # 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
    plot_curve_comparison(
        V_data=V_real, 
        I_real=I_real,
        V_fixed=v_fixed,
        is_transfer=is_transfer,
        mlp_model=pacote_ferramentas,
        yscale="log",
        title=None
    )

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 871us/step


In [6]:
import numpy as np
import pandas as pd
import json
import os
from typing import Tuple    

def load_data_for_inference(csv_path: str) -> Tuple[np.ndarray, np.ndarray]:
    with open(csv_path, 'r') as f:
        infer_data = json.load(f)
    
    for item in infer_data:
        npz_path = item["npz_path"]
        csv_path = item["csv_path"]
        v_fixed = item["fixed_voltage_simulated"]
        params = item["params"]
        
        # Determinar se é curva de Transferência ou Saída
        is_transfer = 'transfer' in os.path.basename(npz_path).lower()
        
        # Carregar dados do CSV
        df_exp3 = pd.read_csv(csv_path)
        V_data = df_exp3.iloc[:, 0].to_numpy()  # Supondo que a primeira coluna seja V
        I_data = df_exp3.iloc[:, 1].to_numpy()  # Supondo que a segunda coluna seja I
    return V_data, I_data, v_fixed, is_transfer

## **EXP 2**

In [ ]:
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(3, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(3, activation='tanh', name='HL2'),
    Dense(3, activation='tanh', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=500, batch_size=128, validation_split=0.2)

#Salvar o modelo treinado e o scaler_X
model.save("/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/modelo_exp2.keras")
joblib.dump(scaler_X, "scaler_X_exp2.pkl")
joblib.dump(Y, "scaler_Y_exp2.pkl")

Epoch 1/10000


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 9.7224 - mae: 1.9733 - val_loss: 8.9016 - val_mae: 1.9384
Epoch 2/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.0927 - mae: 0.9887 - val_loss: 3.3400 - val_mae: 1.1237
Epoch 3/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0377 - mae: 0.5388 - val_loss: 1.1230 - val_mae: 0.6266
Epoch 4/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3308 - mae: 0.3313 - val_loss: 0.3188 - val_mae: 0.3387
Epoch 5/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0974 - mae: 0.2110 - val_loss: 0.0710 - val_mae: 0.1813
Epoch 6/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0333 - mae: 0.1473 - val_loss: 0.0135 - val_mae: 0.0967
Epoch 7/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0191 - mae: 0.1133 - val_loss: 0.0050 - val_mae: 0.0571
Epoch 8/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0160 - mae: 0.1001 - val_loss: 0.0048 - val_mae: 0.0575
Epoch 9/10000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 

['scaler_Y_exp2.pkl']

In [3]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model
import joblib

BASE_PATH_MODEL_EXP2 = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp2"
PATH_MODEL_EXP2 = BASE_PATH_MODEL_EXP2 + '/modelo_exp2.keras'
trained_model_exp2 = load_model(PATH_MODEL_EXP2)
scaler_X_exp2 = joblib.load(BASE_PATH_MODEL_EXP2 + "/scaler_X_exp2.pkl")
scaler_Y_exp2 = joblib.load(BASE_PATH_MODEL_EXP2 + "/scaler_Y_exp2.pkl")

2026-01-26 08:31:54.104319: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-26 08:31:54.156037: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-26 08:31:56.158133: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1769427118.207762 1783080 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1769427118.212609 1783080 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are install

In [6]:
import numpy as np
import json

PATH_TEST_JSON_EXP2 = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
V_real_exp2, I_real_exp2, v_fixed_exp2, is_transfer_exp2 = load_data_for_inference(PATH_TEST_JSON_EXP2)

In [7]:
# 2. Preparar o pacote de ferramentas para plotagem
tools_package = (prepare_mlp_features, trained_model_exp2, scaler_X_exp2)

In [ ]:
# 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
plot_curve_comparison(
    V_data=V_real_exp2, 
    I_real=I_real_exp2,
    V_fixed=v_fixed_exp2,
    is_transfer=is_transfer_exp2,
    mlp_model=tools_package,
    yscale="log",
    title=None
)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


## **EXP 3**

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(3, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(3, activation='tanh', name='HL2'),
    Dense(3, activation='tanh', name='HL3'),
    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=2000, batch_size=128, validation_split=0.2)

#Salvar o modelo treinado e o scaler_X
model.save("/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp3/modelo_exp3.keras")
joblib.dump(scaler_X, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp3/scaler_X_exp3.pkl")
joblib.dump(Y, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp3/scaler_Y_exp3.pkl")

Epoch 1/2000


/home/rsb6/miniconda3/envs/ml_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



688/688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 9.3343 - mae: 1.9209 - val_loss: 8.2874 - val_mae: 1.9737
Epoch 2/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2.9923 - mae: 1.0542 - val_loss: 3.2532 - val_mae: 1.2139
Epoch 3/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0173 - mae: 0.5911 - val_loss: 1.0059 - val_mae: 0.6591
Epoch 4/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2823 - mae: 0.3174 - val_loss: 0.2479 - val_mae: 0.3253
Epoch 5/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0731 - mae: 0.1891 - val_loss: 0.0497 - val_mae: 0.1626
Epoch 6/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0233 - mae: 0.1267 - val_loss: 0.0100 - val_mae: 0.0850
Epoch 7/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0133 - mae: 0.0978 - val_loss: 0.0054 - val_mae: 0.0610
Epoch 8/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0106 - mae: 0.0854 - val_loss: 0.0059 - val_mae: 0.0646
Epoch 9/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

['/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp3/scaler_Y_exp3.pkl']

In [10]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model
import joblib

BASE_PATH_MODEL_EXP3 = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp3"
PATH_MODEL_EXP3 = BASE_PATH_MODEL_EXP3 + '/modelo_exp3.keras'
trained_model_exp3 = load_model(PATH_MODEL_EXP3)
scaler_X_exp3 = joblib.load(BASE_PATH_MODEL_EXP3 + "/scaler_X_exp3.pkl")
scaler_Y_exp3 = joblib.load(BASE_PATH_MODEL_EXP3 + "/scaler_Y_exp3.pkl")

In [11]:
PATH_TEST_JSON = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
V_real_exp3, I_real_exp3, v_fixed_exp3, is_transfer_exp3 = load_data_for_inference(PATH_TEST_JSON)

In [12]:
    # 2. Preparar o pacote de ferramentas para plotagem
    mlp_tools_exp3 = (prepare_mlp_features, trained_model_exp3, scaler_X_exp3)
    
    # 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
    plot_curve_comparison(
        V_data=V_real_exp3, 
        I_real=I_real_exp3,
        V_fixed=v_fixed_exp3,
        is_transfer=is_transfer_exp3,
        mlp_model=mlp_tools_exp3,
        yscale="log",
        title=None
    )

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 869us/step


## **EXP4**

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura atualizada
model = Sequential([
    # Input Layer e HL1: Alterado input_shape para (2,)
    Dense(4, activation='tanh', input_shape=(2,), name='HL1'),
    
    # HL2 e HL3 permanecem iguais
    Dense(4, activation='tanh', name='HL2'),
    Dense(4, activation='tanh', name='HL3'),
    # Dense(4, activation='tanh', name='HL4'),    
    # Output Layer
    Dense(1, activation='linear', name='Output_ln_Id') 
])

# Compilação
model.compile(optimizer='adam', loss='mse', metrics=['mae']) 

# O treinamento agora deve funcionar
# Certifique-se de que o seu X_scaled foi gerado a partir do X com 2 colunas
model.fit(X_scaled, Y, epochs=2000, batch_size=128, validation_split=0.2)

#Salvar o modelo treinado e o scaler_X
model.save("/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp4/modelo_exp4.keras")
joblib.dump(scaler_X, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp4/scaler_X_exp4.pkl")
joblib.dump(Y, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp4/scaler_Y_exp4.pkl")

Epoch 1/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8.2763 - mae: 1.7708 - val_loss: 5.4152 - val_mae: 1.4875
Epoch 2/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.5428 - mae: 0.6626 - val_loss: 1.3750 - val_mae: 0.6836
Epoch 3/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3518 - mae: 0.3241 - val_loss: 0.2766 - val_mae: 0.3022
Epoch 4/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0784 - mae: 0.1916 - val_loss: 0.0417 - val_mae: 0.1323
Epoch 5/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0288 - mae: 0.1377 - val_loss: 0.0060 - val_mae: 0.0619
Epoch 6/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0214 - mae: 0.1143 - val_loss: 0.0015 - val_mae: 0.0313
Epoch 7/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0201 - mae: 0.1049 - val_loss: 9.6618e-04 - val_mae: 0.0263
Epoch 8/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0197 - mae: 0.1021 - val_loss: 0.0014 - val_mae: 0.0305
Epoch 9/2000
688/688 ━━━━━━━━━━━━━━━

['/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp4/scaler_Y_exp4.pkl']

In [14]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model
import joblib

BASE_PATH_MODEL_EXP4 = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp4"
PATH_MODEL_EXP4 = BASE_PATH_MODEL_EXP4 + '/modelo_exp4.keras'
trained_model_exp4 = load_model(PATH_MODEL_EXP4)
scaler_X_exp4 = joblib.load(BASE_PATH_MODEL_EXP4 + "/scaler_X_exp4.pkl")
scaler_Y_exp4 = joblib.load(BASE_PATH_MODEL_EXP4 + "/scaler_Y_exp4.pkl")

In [15]:
PATH_TEST_JSON = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
V_real_exp4, I_real_exp4, v_fixed_exp4, is_transfer_exp4 = load_data_for_inference(PATH_TEST_JSON)

In [16]:
# 2. Preparar o pacote de ferramentas para plotagem
mlp_tools_exp4 = (prepare_mlp_features, trained_model_exp4, scaler_X_exp4)

# 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
plot_curve_comparison(
    V_data=V_real_exp4, 
    I_real=I_real_exp4,
    V_fixed=v_fixed_exp4,
    is_transfer=is_transfer_exp4,
    mlp_model=mlp_tools_exp4,
    yscale="log",
    title=None
)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 987us/step


## **EXP5**

In [28]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from sklearn.preprocessing import StandardScaler
import joblib

# --- 1. NORMALIZAÇÃO ADAPTADA ---
# Escalonador para as entradas [Vg, Vd]
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Escalonador para a saída ln(|Id|) -> CRUCIAL para melhorar o fit
scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y) # Y já deve ser np.log(max(abs(I), 1e-30))

# --- 2. MODELO COM ARQUITETURA APRIMORADA ---
model = Sequential([
    Input(shape=(2,)), # Entrada bruta [Vg, Vd]
    
    # Camadas mais largas e ativação 'silu' (Swish) para melhor não-linearidade
    # Dense(512, activation='silu', name='HL1'),
    Dense(256, activation='silu', name='HL2'),
    Dense(128, activation='silu', name='HL3'),
    # Dense(64, activation='silu', name='HL4'),
    
    # Dropout leve para evitar que o modelo "decore" ruídos do CSV
    Dropout(0.05),
    
    # Saída linear (vai prever o ln(|Id|) escalonado)
    Dense(1, activation='linear', name='Output')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# --- 3. TREINAMENTO ---
history = model.fit(
    X_scaled, Y_scaled, 
    epochs=2000,          # Aumentado para permitir ajuste fino
    batch_size=128, 
    validation_split=0.2,
    verbose=1
)

# --- 4. SALVAR TUDO PARA A INFERÊNCIA ---
model.save("/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp5/modelo_exp5.keras")
joblib.dump(scaler_X, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp5/scaler_X_exp5.pkl")
joblib.dump(scaler_Y, "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp5/scaler_Y_exp5.pkl")

print("Treinamento concluído e scalers salvos.")

Epoch 1/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0301 - mae: 0.1175 - val_loss: 0.0352 - val_mae: 0.1237
Epoch 2/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0144 - mae: 0.0819 - val_loss: 0.0248 - val_mae: 0.1045
Epoch 3/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0112 - mae: 0.0673 - val_loss: 0.0197 - val_mae: 0.0958
Epoch 4/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0091 - mae: 0.0606 - val_loss: 0.0147 - val_mae: 0.0750
Epoch 5/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0072 - mae: 0.0545 - val_loss: 0.0103 - val_mae: 0.0638
Epoch 6/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0053 - mae: 0.0476 - val_loss: 0.0053 - val_mae: 0.0484
Epoch 7/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0035 - mae: 0.0402 - val_loss: 0.0028 - val_mae: 0.0424
Epoch 8/2000
688/688 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0025 - mae: 0.0351 - val_loss: 0.0013 - val_mae: 0.0233
Epoch 9/2000
688/688 ━━━━━━━━━━━━━━━━━━━

In [5]:
def inference_with_y_scaler(v_data, v_fixed, is_transfer, model, scaler_X, scaler_Y):
    # 1. Preparar e Escalar X
    x_raw = prepare_mlp_features(v_data, v_fixed, is_transfer)
    x_scaled = scaler_X.transform(x_raw)
    # 2. Predição (Resultado ainda está na escala do scaler_Y)
    y_pred_scaled = model.predict(x_scaled, verbose=0)
    
    # 3. VOLTAR PARA O ln(|Id|) REAL
    y_pred_ln = scaler_Y.inverse_transform(y_pred_scaled).ravel()
    
    # 4. VOLTAR PARA AMPERES
    i_pred = np.exp(y_pred_ln)
    
    return i_pred

In [15]:
# # Ferramentas necessárias para a inferência completa
# # Nota: Adicionamos o scaler_Y aqui
# mlp_tools = (prepare_mlp_features, model, scaler_X, scaler_Y)

In [20]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import Optional, Tuple, Callable

def plot_curve(
    V_data: np.ndarray, 
    I_real: np.ndarray,
    V_fixed: float,
    is_transfer: bool,
    mlp_model: Optional[Tuple[Callable, object, object, object]] = None,
    yscale: str = "log", 
    title: Optional[str] = None
):
    """
    Plota a curva real e a previsão da MLP, considerando a desnormalização do Target (Y).

    mlp_model : tuple (prepare_X_func, model, scaler_X, scaler_Y)
    """

    fig = go.Figure()
    
    y_scale = yscale.lower()
    is_log_scale = y_scale in ("log", "logarithmic")
    y_axis_title = "Corrente Absoluta (A)" if is_log_scale else "Corrente (A)"
    
    # --- CURVA REAL ---
    I_plot_real = np.abs(I_real)
    if is_log_scale:
        I_plot_real[I_plot_real <= 0] = 1e-30
    
    fig.add_trace(go.Scatter(
        x=V_data, 
        y=I_plot_real, 
        mode='lines+markers', 
        name="Real (Dados CSV)",
        # marker=dict(symbol='circle-open'),
        line=dict(color='blue')
    ))

    # --- PREVISÃO MLP ---
    if mlp_model is not None:
        # 0. Desempacota as 4 ferramentas (X e Y agora têm scalers)
        prepare_X_func, model, scaler_X, scaler_Y = mlp_model
        
        # 1. Preparar features X (2 colunas: Vg, Vd)
        X_test = prepare_X_func(V_data, V_fixed, is_transfer)
        
        # 2. Padronizar X
        X_test_scaled = scaler_X.transform(X_test)
        
        # 3. Prever Y (O resultado sai escalonado entre ~ -1 e 1)
        Y_pred_scaled = model.predict(X_test_scaled, verbose=0)
        
        # 4. DESNORMALIZAR Y (Voltar para a escala do ln(|Id|))
        # O reshape(-1, 1) é necessário para o scaler do scikit-learn
        Y_pred_ln = scaler_Y.inverse_transform(Y_pred_scaled.reshape(-1, 1)).ravel()
        
        # 5. Converter de Logaritmo para Corrente Linear
        I_pred_abs = np.exp(Y_pred_ln)
        
        # 6. Preparar para Plotagem
        I_plot_pred = I_pred_abs.copy()
        if is_log_scale:
            I_plot_pred[I_plot_pred <= 0] = 1e-30 
        
        fig.add_trace(go.Scatter(
            x=V_data, 
            y=I_plot_pred, 
            mode='lines', 
            name="Previsão MLP (Optimized)",
            line=dict(color='red', width=3, dash='dash')
        ))
        
        # Métrica de Erro: MAE no domínio logarítmico
        # Usamos o real em log para comparar com o Y_pred_ln desnormalizado
        I_real_ln = np.log(np.maximum(I_plot_real, 1e-30))
        mae_ln = np.mean(np.abs(I_real_ln - Y_pred_ln))
        
        suffix = f" (MAE_ln={mae_ln:.4f})"
        title = (title if title else f"V_fixed={V_fixed}V") + suffix


    fig.update_layout(
        title=title,
        xaxis_title="V_G (V)" if is_transfer else "V_D (V)",
        yaxis_title=y_axis_title,
        yaxis_type="log" if is_log_scale else "linear",
        template="plotly_dark",
        legend_title="Curvas"
    )

    fig.show()

In [17]:
# Carregar o modelo treinado e os scalers
from tensorflow.keras.models import load_model


BASE_PATH_MODEL_EXP5 = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v3/exp5"
PATH_MODEL_EXP5 = BASE_PATH_MODEL_EXP5 + '/modelo_exp5.keras'
trained_model_exp5 = load_model(PATH_MODEL_EXP5)
scaler_X_exp5 = joblib.load(BASE_PATH_MODEL_EXP5 + "/scaler_X_exp5.pkl")
scaler_Y_exp5 = joblib.load(BASE_PATH_MODEL_EXP5 + "/scaler_Y_exp5.pkl")

In [18]:
PATH_TEST_JSON = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
V_real_exp5, I_real_exp5, v_fixed_exp5, is_transfer_exp5 = load_data_for_inference(PATH_TEST_JSON)

In [21]:

# 3. Chamar a função de plotagem (já ajustada com os 2 scalers)
mlp_tools_exp5 = (prepare_mlp_features, trained_model_exp5, scaler_X_exp5, scaler_Y_exp5)

plot_curve(
    V_data=V_real_exp5, 
    I_real=I_real_exp5,
    V_fixed=v_fixed_exp5,
    is_transfer=is_transfer_exp5,
    mlp_model=mlp_tools_exp5,
    yscale="log",
    title="Teste de Inferência - OTFT"
)